In [2]:
import matplotlib.pyplot as plt
# from erddapClient import ERDDAP_Griddap
from netCDF4 import Dataset
import numpy as np
import xarray as xr
import cmocean
import datetime
import geopandas as gpd
from shapely.ops import linemerge
from shapely.geometry import Polygon
from matplotlib.path import Path

from mpl_toolkits.basemap import Basemap
from matplotlib import colors

In [3]:
# -- Load data --
file_id = Dataset('../../data/chl/chl_1998_2023_l4_month_multi_4k.nc')
ras  = file_id.variables["CHL"][:]
lat  = file_id.variables["latitude"][:]
lon  = file_id.variables["longitude"][:]
time = file_id.variables["time"][:]
file_id.close()

# -- Build date vector --
timedelta_vector = (time * np.timedelta64(1, 'D')).astype('timedelta64[ns]')
base_date   = np.datetime64('1900-01-01')
date_vector = base_date + timedelta_vector

In [5]:
# -- Load and build the Hawaii EEZ polygon --
gdf = gpd.read_file('../../data/eez/USMaritimeLimitsNBoundaries.shp')
hawaii = gdf[gdf['REGION'] == 'Hawaiian Islands']
eez_geom = hawaii[hawaii['EEZ'] == 1.0].geometry.iloc[0]
merged = linemerge(eez_geom)

# Merge parts across the antimeridian
parts = list(merged.geoms)
coords0 = list(parts[0].coords)
coords1 = list(parts[1].coords)
coords1_shifted = [(-360 + x, y) if x > 0 else (x, y) for x, y in coords1]
all_coords = coords0 + coords1_shifted
eez_poly = Polygon(all_coords)

# -- Create a 2D mask on the lat/lon grid --
lon_grid, lat_grid = np.meshgrid(lon, lat)
points = np.column_stack((lon_grid.ravel(), lat_grid.ravel()))

# Use matplotlib Path for fast point-in-polygon test
eez_path = Path(np.array(eez_poly.exterior.coords))
inside = eez_path.contains_points(points).reshape(lon_grid.shape)

# -- Apply mask: set pixels inside the EEZ to NaN --
ras[:, inside] = np.nan

In [6]:
# Extract month number (1-12) for each time step
months = date_vector.astype('datetime64[M]').astype(int) % 12 + 1

# Compute median monthly climatology (12 months, each pixel)
monthly_clim = np.zeros((12, ras.shape[1], ras.shape[2]))
for m in range(1, 13):
    monthly_clim[m - 1, :, :] = np.nanmedian(ras[months == m, :, :], axis=0)

# Subtract the climatology from each time step
ras_anom = np.zeros_like(ras)
for i in range(ras.shape[0]):
    ras_anom[i, :, :] = ras[i, :, :] - monthly_clim[months[i] - 1, :, :]

/tmp/ipykernel_46409/4094170435.py:7: RuntimeWarning: All-NaN slice encountered
  monthly_clim[m - 1, :, :] = np.nanmedian(ras[months == m, :, :], axis=0)


In [7]:
# Compute per-pixel mean and standard deviation across all time steps
pixel_mean = np.nanmean(ras, axis=0)  # shape: (lat, lon)
pixel_std  = np.nanstd(ras, axis=0)   # shape: (lat, lon)
# Create mask: +1 = above mean+1std, -1 = below mean-1std, 0 = within normal range
ras_mask = np.zeros_like(ras)
ras_mask[ras > pixel_mean + pixel_std] =  1
# ras_mask[ras < pixel_mean - pixel_std] = -1

/tmp/ipykernel_46409/497541505.py:2: RuntimeWarning: Mean of empty slice
  pixel_mean = np.nanmean(ras, axis=0)  # shape: (lat, lon)
/home/jamesash/anaconda3/envs/uhdas/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [8]:
ras_extreme = np.where(ras_mask == 1, ras, np.nan)

In [9]:
# Define the 22 start/end date pairs
start_dates = [
    '1998-07-08', '1999-08-01', '2000-08-15', '2001-07-18',
    '2002-06-15', '2003-07-26', '2004-06-03', '2005-07-02', '2006-07-22',
    '2007-08-15', '2008-08-10', '2009-06-13', '2010-07-01', '2011-09-07',
    '2012-09-01', '2013-07-10', '2014-06-14', '2015-08-01', '2016-08-02',
    '2017-07-30', '2018-07-16', '2019-06-11', '2020-08-10', '2021-08-01',
    '2022-07-01'
]

end_dates = [
    '1998-10-06', '1999-10-12', '2000-11-15', '2001-08-04',
    '2002-09-10', '2003-10-08', '2004-08-11', '2005-10-02', '2006-11-11',
    '2007-11-07', '2008-10-22', '2009-10-10', '2010-09-28', '2011-11-12',
    '2012-11-06', '2013-10-02', '2014-09-01', '2015-10-01', '2016-09-25',
    '2017-09-05', '2018-10-27', '2019-09-08', '2020-09-08', '2021-10-01',
    '2022-10-10'
]

# Convert to numpy datetime64
start_dates = np.array(start_dates, dtype='datetime64[D]')
end_dates   = np.array(end_dates, dtype='datetime64[D]')

# Subset and compute grand mean for each date pair
mag_list = []
for s, e in zip(start_dates, end_dates):
    mask = (date_vector >= s) & (date_vector <= e)
    subset = ras_extreme[mask, :, :]
    u = np.nanmean(subset)
    o = np.nanstd(subset)
    mag_list.append(u+2*o) # 2D mean map (lat, lon)

In [12]:
mag_list

[np.float32(0.226158),
 np.float32(0.16235772),
 np.float32(0.2478581),
 np.float32(0.08430518),
 np.float32(0.13964778),
 np.float32(0.40388107),
 np.float32(0.18114346),
 np.float32(0.12035989),
 np.float32(0.14214627),
 np.float32(0.25447083),
 np.float32(0.14470053),
 np.float32(0.17449757),
 np.float32(0.12220141),
 np.float32(0.14587876),
 np.float32(0.16510668),
 np.float32(0.17171189),
 np.float32(0.21506402),
 np.float32(0.09472022),
 np.float32(0.08080868),
 np.float32(0.09931289),
 np.float32(0.16039585),
 np.float32(0.21838583),
 np.float32(0.186324),
 np.float32(0.078868955),
 np.float32(0.14161605)]

In [10]:
from scipy.ndimage import center_of_mass

center_list = []
for s, e in zip(start_dates, end_dates):
    mask = (date_vector >= s) & (date_vector <= e)
    subset_dates = date_vector[mask]
    subset = np.where(np.isnan(ras_extreme[mask, :, :]), 0, ras_extreme[mask, :, :])
    
    t_idx, lat_idx, lon_idx = center_of_mass(subset)
    
    # Convert fractional time index to a date
    t_round = int(round(t_idx))
    t_round = min(t_round, len(subset_dates) - 1)  # safety clamp
    center_date = str(subset_dates[t_round])[:10]   # YYYY-MM-DD
    
    center_list.append({
        'start': str(s),
        'end': str(e),
        'date': center_date,
        'lat': lat[int(round(lat_idx))],
        'lon': lon[int(round(lon_idx))]
    })

In [11]:
for c in center_list:
    print(f"{c['start']} to {c['end']}: date={c['date']}, lat={c['lat']:.2f}, lon={c['lon']:.2f}")

1998-07-08 to 1998-10-06: date=1998-09-01, lat=24.85, lon=-157.65
1999-08-01 to 1999-10-12: date=1999-09-01, lat=26.56, lon=-144.15
2000-08-15 to 2000-11-15: date=2000-10-01, lat=28.06, lon=-150.73
2001-07-18 to 2001-08-04: date=2001-08-01, lat=20.23, lon=-145.69
2002-06-15 to 2002-09-10: date=2002-08-01, lat=26.98, lon=-152.52
2003-07-26 to 2003-10-08: date=2003-09-01, lat=33.31, lon=-152.90
2004-06-03 to 2004-08-11: date=2004-07-01, lat=27.94, lon=-146.48
2005-07-02 to 2005-10-02: date=2005-09-01, lat=22.35, lon=-151.06
2006-07-22 to 2006-11-11: date=2006-10-01, lat=25.40, lon=-148.19
2007-08-15 to 2007-11-07: date=2007-10-01, lat=30.23, lon=-150.48
2008-08-10 to 2008-10-22: date=2008-10-01, lat=24.15, lon=-158.40
2009-06-13 to 2009-10-10: date=2009-08-01, lat=25.02, lon=-159.15
2010-07-01 to 2010-09-28: date=2010-08-01, lat=19.85, lon=-169.44
2011-09-07 to 2011-11-12: date=2011-11-01, lat=24.85, lon=-146.81
2012-09-01 to 2012-11-06: date=2012-10-01, lat=23.31, lon=-157.69
2013-07-10